# Lab G — Fast PyTorch Training, from scratch

**Goal:** learn just enough PyTorch to train a model, then watch that *same* model get **2–3× faster** and use **far less memory** — one lever at a time, measuring every step.

Runs anywhere with a GPU: the **2× RTX 5090 server** or **Google Colab** (Runtime → Change runtime type → **GPU**). It auto-detects your card and scales the workload so it fits.

**How to use it:** run the cells top to bottom. Each one has a short note above it explaining *what* it does and *why*. Read the note, run the cell, look at the output.

The arc:
1. **Tensors** — the basic object
2. **Autograd** — how a model learns
3. **A training loop** — the whole idea, on a tiny model
4. **A realistic workload** — a mini "LLM block stack"
5. **Make it fast** — CPU→GPU, TF32, mixed precision, `torch.compile` (measure each)
6. **Feed the GPU** — the DataLoader
7. **When it won't fit** — gradient checkpointing

## 0 · Setup — what GPU do I have?

First, find out what we're working with. The workload later is scaled from these numbers, so the whole notebook runs on a 16 GB Colab card *or* a 32 GB 5090 without changes.

In [ ]:
import torch, time, math
import torch.nn as nn

print("PyTorch:", torch.__version__)
cuda = torch.cuda.is_available()
print("CUDA available:", cuda)
device = torch.device("cuda" if cuda else "cpu")

total_gb, bf16 = 0.0, False
if cuda:
    p = torch.cuda.get_device_properties(0)
    cap = torch.cuda.get_device_capability(0)
    total_gb = p.total_memory / 1e9
    bf16 = torch.cuda.is_bf16_supported()
    print(f"GPU: {p.name}  (compute {cap[0]}.{cap[1]})")
    print(f"VRAM: {total_gb:.1f} GB    BF16 supported: {bf16}")
else:
    print("No GPU found — turn on a GPU runtime for the speed sections.")

# scale the workload to the card so it runs anywhere
SMALL = (not cuda) or total_gb < 24
print("Workload size:", "SMALL (≈16 GB / Colab)" if SMALL else "FULL (24 GB+)")

## 1 · Tensors — the one object everything is made of

A **tensor** is just a multi-dimensional array (like a NumPy array) that can live on the **GPU** and remember how it was computed (for gradients). Every input, weight, and activation in a model is a tensor.

In [ ]:
x = torch.tensor([[1., 2.],
                  [3., 4.]])
print("tensor:\n", x)
print("shape:", tuple(x.shape), "| dtype:", x.dtype, "| device:", x.device)

# move it onto the GPU — this one call is what makes it fast
xg = x.to(device)
print("moved to:", xg.device)

# tensor math looks just like NumPy
print("matmul x @ x:\n", (xg @ xg))

### The first "it becomes faster": CPU vs GPU

The same matrix multiply, on the CPU vs the GPU. This is *why* we use a GPU at all — thousands of cores doing the multiply-adds in parallel.

In [ ]:
def time_matmul(dev, n, iters):
    a = torch.randn(n, n, device=dev)
    b = torch.randn(n, n, device=dev)
    if dev == "cuda": torch.cuda.synchronize()   # wait for the GPU to actually finish
    t0 = time.perf_counter()
    for _ in range(iters):
        c = a @ b
    if dev == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1000   # ms per matmul

n = 2048
cpu_ms = time_matmul("cpu", n, iters=3)
print(f"CPU  {n}x{n} matmul: {cpu_ms:8.1f} ms")
if cuda:
    gpu_ms = time_matmul("cuda", n, iters=20)
    print(f"GPU  {n}x{n} matmul: {gpu_ms:8.1f} ms")
    print(f"→ the GPU is ~{cpu_ms/gpu_ms:.0f}x faster on this one op")

> **Why `torch.cuda.synchronize()`?** GPU calls are *asynchronous* — Python moves on before the GPU finishes. Without a sync you'd be timing how fast Python *queued* the work, not how fast it *ran*. This is the #1 GPU-timing mistake. We'll use the same idea in the real benchmark below.

## 2 · Autograd — how a model actually learns

Training = "nudge every weight in the direction that lowers the loss." The direction is the **gradient**. PyTorch computes it for you: set `requires_grad=True`, do some math, call `.backward()`, and read `.grad`.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 3*x + 1        # some function of x
y.backward()              # compute dy/dx automatically

print("autograd  dy/dx :", x.grad.item())    # PyTorch did the calculus
print("by hand   2x+3  :", 2*3.0 + 3)         # ...and it matches

That's the whole engine. An optimizer just repeats: compute loss → `backward()` → step each weight a little *down* its gradient → repeat.

## 3 · Your first training loop

Five lines, every time: **zero the grads → forward → loss → backward → step.** Here we teach a tiny network to fit `y = 2x + 1`. Watch the loss fall.

In [ ]:
# synthetic data: a line, plus a little noise
X = torch.randn(1000, 1, device=device)
Y = 2*X + 1 + 0.1*torch.randn_like(X)

model   = nn.Sequential(nn.Linear(1, 64), nn.ReLU(), nn.Linear(64, 1)).to(device)
opt     = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

for step in range(301):
    opt.zero_grad()             # 1 · clear last step's grads
    pred = model(X)             # 2 · forward
    loss = loss_fn(pred, Y)     # 3 · how wrong are we?
    loss.backward()             # 4 · gradients
    opt.step()                  # 5 · nudge the weights
    if step % 60 == 0:
        print(f"step {step:3d}   loss {loss.item():.4f}")

print("final loss:", round(loss.item(), 4))

That loop is *identical* whether the model is a 2-layer toy or a 70-billion-parameter LLM. Everything for the rest of this notebook is about making this loop **run faster** and **fit in less memory** — without changing what it computes.

## 4 · A realistic workload — a mini "LLM block stack"

To see real speedups we need a real, matmul-heavy model. We'll stack a few Transformer encoder layers — the same attention + feed-forward blocks an LLM is built from — and feed it random "token embeddings." The size scales to your GPU.

In [ ]:
import torch.utils.checkpoint as cp

d_model = 512 if SMALL else 1024
nhead   = 8   if SMALL else 16
nlayers = 6   if SMALL else 12
ff      = 4 * d_model
seq     = 256 if SMALL else 512
batch   = 8   if SMALL else 16

class MiniLLM(nn.Module):
    def __init__(self, use_ckpt=False):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model, nhead, ff,
                batch_first=True, norm_first=True, activation="gelu")
            for _ in range(nlayers)
        ])
        self.use_ckpt = use_ckpt
    def forward(self, x):
        for layer in self.layers:
            if self.use_ckpt and self.training:
                x = cp.checkpoint(layer, x, use_reentrant=False)  # recompute in backward
            else:
                x = layer(x)
        return x

def make_batch():
    return torch.randn(batch, seq, d_model, device=device)

params = sum(p.numel() for p in MiniLLM().parameters()) / 1e6
print(f"model: d_model={d_model}, heads={nhead}, layers={nlayers}, seq={seq}, batch={batch}")
print(f"params: {params:.1f}M")

### Measure it *correctly* — a reusable benchmark

Two lessons baked in: **warm up first** (the first steps pay for CUDA init and `torch.compile`), and **synchronize** before reading the clock. We also record **peak memory** so we can watch VRAM drop.

In [ ]:
def make_step(model, opt, amp=False, amp_dtype=torch.bfloat16, scaler=None):
    x = make_batch()                          # fixed batch: we're timing compute, not data
    def step():
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=amp):
            out  = model(x)
            loss = out.float().pow(2).mean()  # dummy scalar loss, enough to backprop
        if scaler is not None:                # FP16 path needs a loss scaler
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:                                 # FP32 / BF16 path
            loss.backward(); opt.step()
    return step

def benchmark(step, iters=20, warmup=8):
    for _ in range(warmup):                   # warmup also triggers compilation
        step()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    e0 = torch.cuda.Event(enable_timing=True)
    e1 = torch.cuda.Event(enable_timing=True)
    e0.record()
    for _ in range(iters):
        step()
    e1.record(); torch.cuda.synchronize()
    ms = e0.elapsed_time(e1) / iters
    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    return ms, peak_mb

assert cuda, "The speed sections need a GPU — enable a GPU runtime and re-run from the top."
print("benchmark ready")

## 5 · Make it fast — one lever at a time

We'll train the *same* MiniLLM four ways and record ms/step and peak MB each time. Start with the honest baseline: **plain FP32, no tricks.**

In [ ]:
results = []

model = MiniLLM().to(device); model.train()
opt   = torch.optim.AdamW(model.parameters(), lr=1e-4)
ms, mb = benchmark(make_step(model, opt, amp=False))
results.append(("FP32 eager", ms, mb))
print(f"FP32 eager      : {ms:7.1f} ms/step   {mb:7.0f} MB")

### Lever 1 — TF32 (free Tensor Cores for FP32 matmuls)

One line tells PyTorch it may run FP32 matmuls on the **Tensor Cores** at slightly reduced internal precision (TF32). Accuracy is basically unchanged; matmuls get much faster. *(No effect on pre-Ampere cards like Colab's T4 — that's expected.)*

In [ ]:
torch.set_float32_matmul_precision("high")   # allow TF32 on matmuls

model = MiniLLM().to(device); model.train()
opt   = torch.optim.AdamW(model.parameters(), lr=1e-4)
ms, mb = benchmark(make_step(model, opt, amp=False))
results.append(("+ TF32", ms, mb))
print(f"+ TF32          : {ms:7.1f} ms/step   {mb:7.0f} MB")

### Lever 2 — Mixed precision (AMP)

Run the heavy math in **16-bit** while keeping the weights in FP32. We pick the dtype from the hardware: **BF16** on modern cards (no loss scaler needed), **FP16 + GradScaler** on older ones — exactly the rule from the lecture. Watch **both** columns move: faster *and* less memory.

In [ ]:
amp_dtype = torch.bfloat16 if bf16 else torch.float16
scaler    = torch.amp.GradScaler() if amp_dtype == torch.float16 else None
print("autocast dtype:", amp_dtype, "| using GradScaler:", scaler is not None)

model = MiniLLM().to(device); model.train()
opt   = torch.optim.AdamW(model.parameters(), lr=1e-4)
ms, mb = benchmark(make_step(model, opt, amp=True, amp_dtype=amp_dtype, scaler=scaler))
results.append(("+ AMP 16-bit", ms, mb))
print(f"+ AMP 16-bit    : {ms:7.1f} ms/step   {mb:7.0f} MB")

### Lever 3 — `torch.compile`

Wrap the model and PyTorch traces it into a graph, **fuses** ops, and generates Triton kernels — the kernel fusion you did by hand, automatic. The **first** step is slow (it compiles); our `warmup` absorbs that, so we time the fast steady state.

In [ ]:
model = MiniLLM().to(device); model.train()
opt   = torch.optim.AdamW(model.parameters(), lr=1e-4)
try:
    cmodel = torch.compile(model)
    ms, mb = benchmark(make_step(cmodel, opt, amp=True, amp_dtype=amp_dtype, scaler=scaler),
                       warmup=12)
    results.append(("+ torch.compile", ms, mb))
    print(f"+ torch.compile : {ms:7.1f} ms/step   {mb:7.0f} MB")
except Exception as e:
    print("torch.compile skipped in this environment:", repr(e)[:140])

### The scoreboard you just built

The same model, the same batch — each lever stacked on the last. This is the W4 "measured wins" slide, generated on *your* GPU.

In [ ]:
base = results[0][1]
print(f"{'setup':<18}{'ms/step':>9}{'speedup':>10}{'peak MB':>10}")
print("-" * 47)
for name, ms, mb in results:
    print(f"{name:<18}{ms:>9.1f}{base/ms:>9.2f}x{mb:>10.0f}")
print("\n(ms lower = faster; speedup vs FP32 baseline; peak MB lower = more room for batch/seq)")

## 6 · Feed the GPU — the DataLoader

A fast GPU is useless if it's waiting for data. The `DataLoader` prepares the next batch on **CPU worker processes** while the GPU trains on the current one. Here we give each sample a little fake CPU work (like decode/tokenize) and compare a **naive** loader vs a **tuned** one.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SynthTokens(Dataset):
    def __init__(self, n=1024): self.n = n
    def __len__(self): return self.n
    def __getitem__(self, i):
        x = torch.randn(seq, d_model)
        for _ in range(30):            # pretend: per-sample CPU work (decode/tokenize/augment)
            x = x * 1.0001
        return x

def time_epoch(loader):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for x in loader:
        x = x.to(device, non_blocking=True)   # async copy (needs pin_memory=True)
    torch.cuda.synchronize()
    return time.perf_counter() - t0

naive = DataLoader(SynthTokens(), batch_size=batch, num_workers=0, pin_memory=False)
tuned = DataLoader(SynthTokens(), batch_size=batch, num_workers=4, pin_memory=True,
                   persistent_workers=True, prefetch_factor=4)

# (run each twice; the first pass pays worker-startup + caches)
time_epoch(naive); time_epoch(tuned)
print(f"naive  (0 workers, no pin) : {time_epoch(naive):.2f} s")
print(f"tuned  (4 workers, pinned) : {time_epoch(tuned):.2f} s")

> With real datasets (image decode, tokenization, disk reads) the gap is **much** bigger than on this synthetic one. The move is always the same: enough `num_workers`, `pin_memory=True`, and `.to(device, non_blocking=True)` so the copy hides behind compute.

## 7 · When it won't fit — gradient checkpointing

Most training memory is **activations** saved for the backward pass. Checkpointing stores them only at layer boundaries and **recomputes** the rest during backward: a big memory cut for a little extra compute. It's how you fit longer sequences / bigger batches — critical for long-context LLMs.

In [ ]:
def run(use_ckpt):
    m = MiniLLM(use_ckpt=use_ckpt).to(device); m.train()
    o = torch.optim.AdamW(m.parameters(), lr=1e-4)
    return benchmark(make_step(m, o, amp=True, amp_dtype=amp_dtype, scaler=scaler),
                     iters=10, warmup=4)

ms_no, mb_no = run(False)
ms_ck, mb_ck = run(True)
print(f"no checkpointing : {ms_no:7.1f} ms   {mb_no:7.0f} MB")
print(f"checkpointing    : {ms_ck:7.1f} ms   {mb_ck:7.0f} MB")
print(f"→ memory saved: {100*(1-mb_ck/mb_no):4.0f}%    time cost: +{100*(ms_ck/ms_no-1):.0f}%")

## What you just proved

- A model is **tensors** + **autograd** + a **5-line loop** — the same loop from a toy net to an LLM.
- The same loop got **2–3× faster** and lighter with a few lines: **TF32 → AMP → `torch.compile`**, plus a **DataLoader** that keeps the GPU fed and **checkpointing** when memory runs out.
- The method never changes: **measure → change one thing → measure → compare.** Never assume.

**The one wall you can't cross here:** a model that won't fit on a single card, or a run that's just too slow alone. That's the next topic — **multi-GPU training (DDP · FSDP · NCCL)**, which turns the Kimi K3 parallelism ideas into code you run on the two 5090s.